In [0]:
import numpy as np
import pydeck as pdk

route = spark.table("workspace.spb.gold_route_nodes").orderBy("seq").toPandas()
bars = spark.table("workspace.spb.silver_bars").select("lon", "lat", "name", "h3_cell").toPandas()

# 1. density layer: bars per H3 cell, shaded yellow to red
cells = bars.groupby("h3_cell").size().rename("n_bars").reset_index()
t = cells.n_bars / cells.n_bars.max()
cells["color"] = [[255, int(220 * (1 - x)), 0, 110] for x in t]

# 2. route: one path per leg, colored from blue (start) to red (end)
def leg_color(i, n):
    x = i / max(n - 1, 1)
    return [int(255 * x), int(80 + 100 * (1 - x)), int(255 * (1 - x))]

n_legs = route.leg_no.max() + 1
paths = [{"leg": int(leg),
          "path": g.sort_values("seq")[["lon", "lat"]].to_numpy().tolist(),
          "color": leg_color(leg, n_legs)}
         for leg, g in route.groupby("leg_no")]

deck = pdk.Deck(
    layers=[
        pdk.Layer("H3HexagonLayer", cells, get_hexagon="h3_cell", get_fill_color="color",
                  extruded=False, pickable=True),
        pdk.Layer("PathLayer", paths, get_path="path", get_color="color",
                  width_min_pixels=2, pickable=True),
        pdk.Layer("ScatterplotLayer", bars, get_position="[lon, lat]", get_radius=25,
                  get_fill_color=[255, 255, 255, 200]),
    ],
    initial_view_state=pdk.ViewState(longitude=30.32, latitude=59.93, zoom=10),
    map_style="dark",
)
displayHTML(deck.to_html(as_string=True))